In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

# --- Paths ---
PROJECT_ROOT = "/content/drive/MyDrive/credit-risk"
DATA_PROCESSED = f"{PROJECT_ROOT}/data/processed"
MODELS_PATH    = f"{PROJECT_ROOT}/models"
FIGURES_PATH   = f"{PROJECT_ROOT}/reports/figures"
SRC_PATH       = f"{PROJECT_ROOT}/src"

sys.path.insert(0, SRC_PATH)

print("✅ Imports and paths ready")

✅ Imports and paths ready


In [3]:
df = pd.read_csv(f"{DATA_PROCESSED}/df_selected.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nTarget distribution:")
print(df['SeriousDlqin2yrs'].value_counts(normalize=True).round(4))

Shape: (150000, 12)

Columns: ['RevolvingUtilizationOfUnsecuredLines', 'age', 'DebtRatio', 'NumberOfOpenCreditLinesAndLoans', 'NumberRealEstateLoansOrLines', 'NumberOfDependents', 'income_missing', 'DebtToIncomeRatio', 'IncomePerDependent', 'TotalLatePayments', 'LatePaymentRatio', 'SeriousDlqin2yrs']

Target distribution:
SeriousDlqin2yrs
0    0.9332
1    0.0668
Name: proportion, dtype: float64


In [4]:
X = df.drop(columns=['SeriousDlqin2yrs'])
y = df['SeriousDlqin2yrs']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Defaults in y:", y.sum(), f"({y.mean()*100:.2f}%)")

X shape: (150000, 11)
y shape: (150000,)
Defaults in y: 10026 (6.68%)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # preserves the 6.68% ratio in both splits
)

print("Train size:", X_train.shape)
print("Test size: ", X_test.shape)
print(f"\nDefaults in train: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Defaults in test:  {y_test.sum()} ({y_test.mean()*100:.2f}%)")

Train size: (120000, 11)
Test size:  (30000, 11)

Defaults in train: 8021 (6.68%)
Defaults in test:  2005 (6.68%)


In [6]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", X_train.shape, "| Defaults:", y_train.sum())
print("After SMOTE: ", X_train_sm.shape, "| Defaults:", y_train_sm.sum())
print(f"\nClass balance after SMOTE:")
print(pd.Series(y_train_sm).value_counts(normalize=True).round(4))

Before SMOTE: (120000, 11) | Defaults: 8021
After SMOTE:  (223958, 11) | Defaults: 111979

Class balance after SMOTE:
SeriousDlqin2yrs
0    0.5
1    0.5
Name: proportion, dtype: float64


In [7]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report
)

def evaluate_model(name, model, X_test, y_test):
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc  = average_precision_score(y_test, y_proba)
    f1      = f1_score(y_test, y_pred)

    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(f"  ROC-AUC : {roc_auc:.4f}")
    print(f"  PR-AUC  : {pr_auc:.4f}  ← primary metric")
    print(f"  F1      : {f1:.4f}")
    print(f"\n{classification_report(y_test, y_pred)}")

    return {"Model": name, "ROC-AUC": roc_auc, "PR-AUC": pr_auc, "F1": f1}

In [8]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric='aucpr',
    verbosity=0
)

xgb_model.fit(X_train_sm, y_train_sm)

xgb_results = evaluate_model("XGBoost Baseline", xgb_model, X_test, y_test)


  XGBoost Baseline
  ROC-AUC : 0.8506
  PR-AUC  : 0.3383  ← primary metric
  F1      : 0.3874

              precision    recall  f1-score   support

           0       0.95      0.96      0.96     27995
           1       0.41      0.37      0.39      2005

    accuracy                           0.92     30000
   macro avg       0.68      0.66      0.67     30000
weighted avg       0.92      0.92      0.92     30000



In [9]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    verbosity=-1
)

lgbm_model.fit(X_train_sm, y_train_sm)

lgbm_results = evaluate_model("LightGBM Baseline", lgbm_model, X_test, y_test)


  LightGBM Baseline
  ROC-AUC : 0.8505
  PR-AUC  : 0.3358  ← primary metric
  F1      : 0.3728

              precision    recall  f1-score   support

           0       0.95      0.97      0.96     27995
           1       0.43      0.33      0.37      2005

    accuracy                           0.93     30000
   macro avg       0.69      0.65      0.67     30000
weighted avg       0.92      0.93      0.92     30000



In [12]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=0
)

cat_model.fit(X_train_sm, y_train_sm)

cat_results = evaluate_model("CatBoost Baseline", cat_model, X_test, y_test)


  CatBoost Baseline
  ROC-AUC : 0.8534
  PR-AUC  : 0.3395  ← primary metric
  F1      : 0.3976

              precision    recall  f1-score   support

           0       0.96      0.96      0.96     27995
           1       0.41      0.39      0.40      2005

    accuracy                           0.92     30000
   macro avg       0.68      0.67      0.68     30000
weighted avg       0.92      0.92      0.92     30000



In [13]:
results = pd.DataFrame([xgb_results, lgbm_results, cat_results])
results = results.sort_values("PR-AUC", ascending=False).reset_index(drop=True)

print("\n📊 Baseline Model Comparison (sorted by PR-AUC)\n")
print(results.to_string(index=False))


📊 Baseline Model Comparison (sorted by PR-AUC)

            Model  ROC-AUC   PR-AUC       F1
CatBoost Baseline 0.853354 0.339503 0.397634
 XGBoost Baseline 0.850634 0.338289 0.387404
LightGBM Baseline 0.850492 0.335837 0.372824


In [14]:
results.to_csv(f"{DATA_PROCESSED}/baseline_results.csv", index=False)
print("✅ Baseline results saved to data/processed/baseline_results.csv")

✅ Baseline results saved to data/processed/baseline_results.csv


In [16]:
from sklearn.metrics import precision_recall_curve

fig, ax = plt.subplots(figsize=(8, 6))

models = [
    ("CatBoost Baseline", cat_model,  "green"),
    ("XGBoost Baseline",  xgb_model,  "blue"),
    ("LightGBM Baseline", lgbm_model, "orange"),
]

for name, model, color in models:
    y_proba = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    ax.plot(recall, precision, label=f"{name} (PR-AUC={pr_auc:.4f})", color=color)

# baseline: random classifier on imbalanced data
ax.axhline(y=y_test.mean(), color='red', linestyle='--', label=f"Random baseline ({y_test.mean():.4f})")

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — Baseline Models")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_PATH}/pr_curves.png", dpi=150)
plt.show()
print("✅ PR curve saved to reports/figures/phase5_pr_curves.png")

✅ PR curve saved to reports/figures/phase5_pr_curves.png
